In [26]:
import pandas as pd

file_path = "data/CIDDS-001-Sampled-Data.parquet"
df = pd.read_parquet(file_path)

df.head()

,Date first seen,Duration,Proto,Src IP Addr,Src Pt,Dst IP Addr,Dst Pt,Packets,Bytes,Flows,Flags,Tos,class,attackType,attackID,attackDescription
0,2017-03-17 14:18:05.000,0.004,TCP,192.168.220.16,34242,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80
1,2017-03-17 14:18:05.000,0.003,TCP,192.168.220.16,34242,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80
2,2017-03-17 14:18:05.001,0.001,TCP,192.168.100.6,80,192.168.220.16,34242.0,4,272,1,.A..SF,0,victim,dos,18,10000 connections on 192.168.100.6:80
3,2017-03-17 14:18:05.001,0.003,TCP,192.168.100.6,80,192.168.220.16,34242.0,4,272,1,.A..SF,0,victim,dos,18,10000 connections on 192.168.100.6:80
4,2017-03-17 14:18:05.002,0.005,TCP,192.168.220.16,34243,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80


In [27]:
df.describe()

,Date first seen,Duration,Src Pt,Dst Pt,Packets,Flows,Tos
count,2619143,2.619143e+06,2.619143e+06,2.619143e+06,2.619143e+06,2619143.0,2.619143e+06
mean,2017-03-19 13:23:47.036103,3.160933e+00,2.435068e+04,2.411023e+04,1.390957e+01,1.0,8.764968e+00
min,2017-03-17 14:18:05,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.0,0.000000e+00
25%,2017-03-18 07:50:54.495500,0.000000e+00,8.000000e+01,8.000000e+01,1.000000e+00,1.0,0.000000e+00
50%,2017-03-20 09:10:24.414000,1.000000e-03,8.082000e+03,8.000000e+03,2.000000e+00,1.0,0.000000e+00
75%,2017-03-20 12:25:55.927000,3.200000e-02,4.966400e+04,4.961800e+04,4.000000e+00,1.0,3.200000e+01
max,2017-03-20 17:42:17.937000,2.745949e+04,6.553500e+04,6.553500e+04,2.034670e+05,1.0,1.920000e+02
std,NaN,2.111605e+02,2.477470e+04,2.478189e+04,9.312207e+02,0.0,1.477747e+01


In [11]:
print(df.columns)

# Or as a list
print(list(df.columns))

Index(['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt',
       'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos',
       'class', 'attackType', 'attackID', 'attackDescription'],
      dtype='str')
['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos', 'class', 'attackType', 'attackID', 'attackDescription']


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, LabelEncoder

# Giả sử df là dữ liệu bạn đã load
# df = pd.read_parquet("data/CIDDS-001-Sampled-Data.parquet")

print("1. Đang làm sạch cột 'Bytes'...")
def clean_bytes(val):
    if pd.isna(val):
        return 0.0
    val = str(val).strip().upper()
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

# Cập nhật CHÍNH XÁC tên 10 cột đặc trưng dựa trên file thực tế của bạn
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 
    'Proto', 'Flags', 'Duration', 'Bytes', 'Packets'
]

# Tách features (X) và target (y)
X = df[features].copy()
y = df['attackType'].copy()

print("2. Đang chia tập Train/Test (Split 70/30)...")
# Stratify theo y để giữ nguyên tỷ lệ các loại tấn công
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("3. Đang mã hóa các biến phân loại (Categorical Encoding)...")
# Xác định các cột dạng chuỗi (thường là IP, Proto, Flags)
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# Fit TRÊN TẬP TRAIN để chống Data Leakage
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# Chuyển toàn bộ X về float để chuẩn bị cho bước Scale
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# Mã hóa Label (attackType)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...")
# Fit scaler CHỈ TRÊN TẬP TRAIN
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nHoàn tất! Kích thước dữ liệu:")
print(f"Train set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Các lớp tấn công đã encode: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

1. Đang làm sạch cột 'Bytes'...
2. Đang chia tập Train/Test (Split 70/30)...
3. Đang mã hóa các biến phân loại (Categorical Encoding)...
4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...

Hoàn tất! Kích thước dữ liệu:
Train set: (1833400, 9)
Test set: (785743, 9)
Các lớp tấn công đã encode: {'---': np.int64(0), 'bruteForce': np.int64(1), 'dos': np.int64(2), 'pingScan': np.int64(3), 'portScan': np.int64(4)}


In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score

# ==========================================
# PHẦN 1: TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING)
# ==========================================

# Giả sử df là dữ liệu bạn đã load
df = pd.read_parquet("data/CIDDS-001-Sampled-Data.parquet")

print("1. Đang làm sạch cột 'Bytes'...")
def clean_bytes(val):
    if pd.isna(val):
        return 0.0
    val = str(val).strip().upper()
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

# Khai báo features (X) và target (y)
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 
    'Proto', 'Flags', 'Duration', 'Bytes', 'Packets'
]
X = df[features].copy()
y = df['attackType'].copy()

print("2. Đang chia tập Train/Test (Split 70/30)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("3. Đang mã hóa Đặc trưng (Frequency) và Nhãn (Label)...")
# A. Frequency Encoding cho X (Đầu vào)
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

# Ép kiểu float chuẩn bị cho Scaling
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# B. Label Encoding cho y (Đầu ra)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n[+] Hoàn tất Tiền xử lý! Kích thước dữ liệu:")
print(f" - Train set: {X_train_scaled.shape}")
print(f" - Test set: {X_test_scaled.shape}")
print(f" - Từ điển Nhãn (y): {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


# ==========================================
# PHẦN 2: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH
# ==========================================
print("\n--- 5. ĐANG TRAIN LẠI MÔ HÌNH ---")

# Random Forest: Đã tăng n_estimators và thêm class_weight='balanced'
rf_model_tuned = RandomForestClassifier(
    n_estimators=10,            # Tăng từ 10 lên 100 cây để chống nhiễu
    criterion='entropy',            
    min_samples_split=2,         
    min_samples_leaf=1,          
    max_features='sqrt',         
    class_weight=None,     # Cứu cánh cho các lớp thiểu số
    random_state=42,             
    n_jobs=-1                    
)
rf_model_tuned.fit(X_train_scaled, y_train)
rf_preds_tuned = rf_model_tuned.predict(X_test_scaled)

# KNN giữ nguyên tham số
knn_model_tuned = KNeighborsClassifier(
    n_neighbors=3, weights='distance', leaf_size=30, metric='minkowski', n_jobs=-1
)
knn_model_tuned.fit(X_train_scaled, y_train)
knn_preds_tuned = knn_model_tuned.predict(X_test_scaled)

print("\n--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---")
print(f"Random Forest F1-Score: {f1_score(y_test, rf_preds_tuned, average='macro'):.4f}")
print(f"KNN F1-Score:           {f1_score(y_test, knn_preds_tuned, average='macro'):.4f}")

print("\n--- CHI TIẾT BÁO CÁO CỦA RANDOM FOREST ---")
print(classification_report(y_test, rf_preds_tuned, target_names=label_encoder.classes_, digits=4, zero_division=0))

1. Đang làm sạch cột 'Bytes'...
2. Đang chia tập Train/Test (Split 70/30)...
3. Đang mã hóa Đặc trưng (Frequency) và Nhãn (Label)...
4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...

[+] Hoàn tất Tiền xử lý! Kích thước dữ liệu:
 - Train set: (1833400, 9)
 - Test set: (785743, 9)
 - Từ điển Nhãn (y): {'---': np.int64(0), 'bruteForce': np.int64(1), 'dos': np.int64(2), 'pingScan': np.int64(3), 'portScan': np.int64(4)}

--- 5. ĐANG TRAIN LẠI MÔ HÌNH ---

--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---
Random Forest F1-Score: 0.9295
KNN F1-Score:           0.9254

--- CHI TIẾT BÁO CÁO CỦA RANDOM FOREST ---
              precision    recall  f1-score   support

         ---     0.9999    1.0000    1.0000    652723
  bruteForce     1.0000    0.9683    0.9839       379
         dos     1.0000    0.9999    1.0000    117280
    pingScan     0.8161    0.5687    0.6703       320
    portScan     0.9909    0.9959    0.9934     15041

    accuracy                         0.9997    785743
   macro av

In [17]:
import pandas as pd

# 1. Tạo DataFrame chứa Nhãn thực tế và Nhãn dự đoán
results_df = pd.DataFrame({
    'Actual': label_encoder.inverse_transform(y_test),
    'Predicted': label_encoder.inverse_transform(rf_preds_tuned)
})

print("\n" + "="*50)
print(" CHI TIẾT PHÂN TÍCH LỖI CHO LỚP 'pingScan'")
print("="*50)

# ==========================================
# PHẦN 1: BỎ LỌT (False Negatives - Recall thấp)
# ==========================================
missed_pingscan = results_df[(results_df['Actual'] == 'pingScan') & (results_df['Predicted'] != 'pingScan')]
total_missed = len(missed_pingscan)

print(f"\n[1] BỎ LỌT: Có {total_missed} trường hợp thực tế là 'pingScan' nhưng bị đoán sai.")
if total_missed > 0:
    print("    Cụ thể, mô hình đã đoán nhầm chúng thành các lớp sau:")
    # Lặp qua từng nhãn để in toàn bộ, không bị truncate (ẩn bớt)
    for label, count in missed_pingscan['Predicted'].value_counts().items():
        print(f"    + Đoán nhầm thành '{label}': {count} dòng")
else:
    print("    Tuyệt vời! Không bỏ lọt trường hợp nào.")

print("-" * 50)

# ==========================================
# PHẦN 2: NHẬN VƠ (False Positives - Precision bị ảnh hưởng)
# ==========================================
fake_pingscan = results_df[(results_df['Actual'] != 'pingScan') & (results_df['Predicted'] == 'pingScan')]
total_fake = len(fake_pingscan)

print(f"\n[2] NHẬN VƠ: Có {total_fake} trường hợp KHÔNG PHẢI 'pingScan' nhưng bị nhận vơ là pingScan.")
if total_fake > 0:
    print("    Thực chất bản chất thật của chúng là:")
    for label, count in fake_pingscan['Actual'].value_counts().items():
        print(f"    + Vốn dĩ là '{label}': {count} dòng")
else:
    print("    Tuyệt vời! Không có trường hợp nhận vơ nào.")

print("\n" + "="*50)


 CHI TIẾT PHÂN TÍCH LỖI CHO LỚP 'pingScan'

[1] BỎ LỌT: Có 139 trường hợp thực tế là 'pingScan' nhưng bị đoán sai.
    Cụ thể, mô hình đã đoán nhầm chúng thành các lớp sau:
    + Đoán nhầm thành 'portScan': 132 dòng
    + Đoán nhầm thành '---': 6 dòng
    + Đoán nhầm thành 'bruteForce': 1 dòng
--------------------------------------------------

[2] NHẬN VƠ: Có 44 trường hợp KHÔNG PHẢI 'pingScan' nhưng bị nhận vơ là pingScan.
    Thực chất bản chất thật của chúng là:
    + Vốn dĩ là 'portScan': 41 dòng
    + Vốn dĩ là '---': 2 dòng
    + Vốn dĩ là 'bruteForce': 1 dòng



In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("=== BƯỚC 1: ĐỌC DỮ LIỆU ===")
# Bạn nhớ đảm bảo đường dẫn file này khớp với thư mục của bạn nhé
file_path = "data/CIDDS-001-Sampled-Data.parquet" 
df = pd.read_parquet(file_path)
print(f"[+] Đã load thành công {df.shape[0]} dòng và {df.shape[1]} cột.")

print("\n=== BƯỚC 2: TIỀN XỬ LÝ VÀ TẠO ĐẶC TRƯNG (FEATURE ENGINEERING) ===")
# 2.1 Làm sạch cột Bytes
def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

# 2.2 Tạo đặc trưng mới: Cờ báo hiệu Port = 0 (Chuyên trị Ping/ICMP)
df['is_Port_Zero'] = ((df['Dst Pt'] == 0) | (df['Dst Pt'] == '0') | df['Dst Pt'].isna()).astype(int)

# 2.3 Chốt danh sách Đặc trưng (X) và Nhãn (y)
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 
    'Proto', 'Flags', 'Duration', 'Bytes', 'Packets', 'is_Port_Zero'
]
X = df[features].copy()
y = df['attackType'].copy()

print("\n=== BƯỚC 3: CHIA TẬP DỮ LIỆU (TRAIN/TEST SPLIT) ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"[+] Tập Train: {X_train.shape[0]} dòng")
print(f"[+] Tập Test : {X_test.shape[0]} dòng")

print("\n=== BƯỚC 4: MÃ HÓA THÔNG MINH (ENCODING) ===")
# Phân loại các cột dạng chuỗi (Categorical)
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# 4.1. Mã hóa IP (Dùng Frequency Encoding)
freq_cols = [col for col in cat_cols if 'IP' in col]
print(f" -> Đang mã hóa theo Tần suất cho các cột: {freq_cols}")
for col in freq_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

# 4.2. Mã hóa Protocol, Flags, Port (Dùng Ordinal Encoding)
ord_cols = [col for col in cat_cols if col not in freq_cols]
if ord_cols:
    print(f" -> Đang mã hóa dạng Phân lớp (Ordinal) cho các cột: {ord_cols}")
    ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[ord_cols] = ordinal_encoder.fit_transform(X_train[ord_cols])
    X_test[ord_cols] = ordinal_encoder.transform(X_test[ord_cols])

# 4.3. Ép kiểu về Float cho đồng nhất
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# 4.4. Mã hóa Nhãn đích (Label y)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("\n=== BƯỚC 5: CHUẨN HÓA DỮ LIỆU (SCALING) ===")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("[+] Hoàn tất chuẩn hóa.")

print("\n=== BƯỚC 6: HUẤN LUYỆN MÔ HÌNH RANDOM FOREST ===")
# Sử dụng 100 cây và class_weight=None theo bài kiểm tra thực tế
rf_model_tuned = RandomForestClassifier(
    n_estimators=100,            
    criterion='gini',            
    max_features='sqrt',         
    class_weight=None,         
    random_state=42,             
    n_jobs=-1                    
)
rf_model_tuned.fit(X_train_scaled, y_train)
rf_preds_tuned = rf_model_tuned.predict(X_test_scaled)
print("[+] Huấn luyện thành công!")

print("\n================== BÁO CÁO TỔNG QUAN ==================")
print(classification_report(y_test, rf_preds_tuned, target_names=label_encoder.classes_, digits=4, zero_division=0))

print("\n=== BƯỚC 7: MỔ XẺ LỖI CỦA LỚP PINGSCAN ===")
results_df = pd.DataFrame({
    'Actual': label_encoder.inverse_transform(y_test),
    'Predicted': label_encoder.inverse_transform(rf_preds_tuned)
})

missed_pingscan = results_df[(results_df['Actual'] == 'pingScan') & (results_df['Predicted'] != 'pingScan')]
print(f"[1] BỎ LỌT (False Negatives - Recall bị giảm): {len(missed_pingscan)} trường hợp.")
if len(missed_pingscan) > 0:
    for label, count in missed_pingscan['Predicted'].value_counts().items():
        print(f"    + Nhầm thành '{label}': {count} dòng")

fake_pingscan = results_df[(results_df['Actual'] != 'pingScan') & (results_df['Predicted'] == 'pingScan')]
print(f"\n[2] NHẬN VƠ (False Positives - Precision bị giảm): {len(fake_pingscan)} trường hợp.")
if len(fake_pingscan) > 0:
    for label, count in fake_pingscan['Actual'].value_counts().items():
        print(f"    + Vốn dĩ là '{label}': {count} dòng")
print("=========================================================")

=== BƯỚC 1: ĐỌC DỮ LIỆU ===
[+] Đã load thành công 2619143 dòng và 16 cột.

=== BƯỚC 2: TIỀN XỬ LÝ VÀ TẠO ĐẶC TRƯNG (FEATURE ENGINEERING) ===

=== BƯỚC 3: CHIA TẬP DỮ LIỆU (TRAIN/TEST SPLIT) ===
[+] Tập Train: 1833400 dòng
[+] Tập Test : 785743 dòng

=== BƯỚC 4: MÃ HÓA THÔNG MINH (ENCODING) ===
 -> Đang mã hóa theo Tần suất cho các cột: ['Src IP Addr', 'Dst IP Addr']
 -> Đang mã hóa dạng Phân lớp (Ordinal) cho các cột: ['Proto', 'Flags']

=== BƯỚC 5: CHUẨN HÓA DỮ LIỆU (SCALING) ===
[+] Hoàn tất chuẩn hóa.

=== BƯỚC 6: HUẤN LUYỆN MÔ HÌNH RANDOM FOREST ===
[+] Huấn luyện thành công!

================== BÁO CÁO TỔNG QUAN ==================
              precision    recall  f1-score   support

         ---     0.9999    1.0000    1.0000    652723
  bruteForce     0.9973    0.9683    0.9826       379
         dos     1.0000    0.9999    1.0000    117280
    pingScan     0.7854    0.5719    0.6618       320
    portScan     0.9910    0.9957    0.9933     15041

    accuracy                 

In [13]:
import pandas as pd

# 1. Tạo Series chứa dự đoán và nhãn thực tế, GẮN KÈM INDEX gốc của X_test
predicted_labels = pd.Series(label_encoder.inverse_transform(rf_preds_tuned), index=X_test.index)
actual_labels = pd.Series(label_encoder.inverse_transform(y_test), index=X_test.index)

# 2. Lấy Index của các dòng dự đoán là pingScan và portScan
idx_pred_ping = predicted_labels[predicted_labels == 'pingScan'].head(50).index
idx_pred_port = predicted_labels[predicted_labels == 'portScan'].head(50).index

# 3. Tạo một DataFrame tổng hợp từ tập gốc (df) để hiển thị chữ dễ đọc
# Kèm theo 2 cột: "Thực tế là" và "Mô hình đoán là"
df_display = df.loc[:, features].copy()
df_display['Thực tế là'] = actual_labels
df_display['Mô hình đoán là'] = predicted_labels

print("\n" + "="*80)
print(" 🕵️‍♂️ TOP 50 DÒNG MÔ HÌNH DỰ ĐOÁN LÀ: PING SCAN")
print("="*80)
# Dùng .to_string() để ép in ra hết các cột, không bị ẩn
print(df_display.loc[idx_pred_ping].to_string())


print("\n\n" + "="*80)
print(" 🕵️‍♂️ TOP 50 DÒNG MÔ HÌNH DỰ ĐOÁN LÀ: PORT SCAN")
print("="*80)
print(df_display.loc[idx_pred_port].to_string())


 🕵️‍♂️ TOP 50 DÒNG MÔ HÌNH DỰ ĐOÁN LÀ: PING SCAN
            Src IP Addr  Src Pt      Dst IP Addr   Dst Pt  Proto   Flags  Duration    Bytes  Packets  is_Port_Zero Thực tế là Mô hình đoán là
1441813  192.168.220.16       0   192.168.200.87      8.0  ICMP   ......     0.000     42.0        1             0   pingScan        pingScan
1323420  192.168.220.16       0  192.168.210.234      8.0  ICMP   ......     0.426     84.0        2             0   pingScan        pingScan
1445433  192.168.220.16       0   192.168.200.20      8.0  ICMP   ......     0.000     42.0        1             0   pingScan        pingScan
2100582  192.168.220.16       0   192.168.210.50      8.0  ICMP   ......     0.000     42.0        1             0   pingScan        pingScan
1323880  192.168.220.16       0  192.168.210.240      8.0  ICMP   ......     0.584     84.0        2             0   pingScan        pingScan
1448040   192.168.100.6      80   192.168.220.16  56358.0  TCP    .A...F     0.004    132.0       

In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

print("======================================================")
print("   BẮT ĐẦU CHẠY FULL DATA: CHỈ TEST RANDOM FOREST     ")
print("======================================================")

# ==========================================
# 1. LOAD VÀ TIỀN XỬ LÝ CHUNG
# ==========================================
file_path = "data/CIDDS-001-Sampled-Data.parquet"
df = pd.read_parquet(file_path)
print(f"-> Đã nạp thành công {len(df)} dòng dữ liệu gốc.")

def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

del df
gc.collect()

# ==========================================
# 2. CHIA TẬP TRAIN/TEST VÀ CHUẨN HÓA
# ==========================================
print("\n-> Đang chia tập Train/Test theo thời gian...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

print("-> Đang thực hiện Frequency Encoding và Min-Max Scaling...")
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

X_train, X_test = X_train.astype(np.float32), X_test.astype(np.float32) 
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# ==========================================
# 3. TẠO CHUỖI MULTI-FLOW TỐI ƯU
# ==========================================
# BẠN CÓ THỂ ĐỔI THÀNH 70 Ở ĐÂY ĐỂ KIỂM CHỨNG SỰ TỤT DỐC CỦA RF
window_rf = 10 
print(f"\n-> Đang tạo ma trận chuỗi 3D cho RF (Window = {window_rf})...")

def create_sequences_optimized(X_data, y_data, window_size):
    n_samples = len(X_data) - window_size + 1
    X_seq = np.zeros((n_samples, window_size, X_data.shape[1]), dtype=np.float32)
    y_seq = np.zeros((n_samples,), dtype=y_data.dtype)
    for i in range(window_size - 1, len(X_data)):
        idx = i - window_size + 1
        X_seq[idx] = X_data[idx : i + 1] 
        y_seq[idx] = y_data[i]           
    return X_seq, y_seq

X_train_seq_rf, y_train_seq_rf = create_sequences_optimized(X_train_scaled, y_train_encoded, window_rf)
X_test_seq_rf, y_test_seq_rf = create_sequences_optimized(X_test_scaled, y_test_encoded, window_rf)

# ==========================================
# 4. HUẤN LUYỆN MÔ HÌNH RANDOM FOREST
# ==========================================
print("\n======================================================")
print(f"   HUẤN LUYỆN RANDOM FOREST (LÀM PHẲNG WINDOW = {window_rf})")
print("======================================================")

# Làm phẳng mảng 3D thành 2D để đưa vào RF
X_train_flat = X_train_seq_rf.reshape(X_train_seq_rf.shape[0], -1)
X_test_flat = X_test_seq_rf.reshape(X_test_seq_rf.shape[0], -1)

# Cấu hình RF chuẩn theo Paper 2
rf_multi = RandomForestClassifier(
    n_estimators=100, 
    max_depth=35, 
    max_features=100, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=42
)

print("-> Đang huấn luyện Random Forest... (Vui lòng chờ vài phút)")
rf_multi.fit(X_train_flat, y_train_seq_rf)

print("-> Đang dự đoán tập Test...")
rf_multi_preds = rf_multi.predict(X_test_flat)

# Tìm các nhãn thực sự có trong tập test của chuỗi
labels_in_test_rf = np.unique(y_test_seq_rf)
names_in_test_rf = label_encoder.inverse_transform(labels_in_test_rf)

print(f"\n=> KẾT QUẢ: F1-Score Random Forest (Window {window_rf}): {f1_score(y_test_seq_rf, rf_multi_preds, average='macro'):.4f}")
print(f"\n[BẢNG CHI TIẾT TỪNG LỚP TẤN CÔNG (WINDOW {window_rf})]")
print(classification_report(y_test_seq_rf, rf_multi_preds, labels=labels_in_test_rf, target_names=names_in_test_rf, digits=4, zero_division=0))

   BẮT ĐẦU CHẠY FULL DATA: CHỈ TEST RANDOM FOREST     
-> Đã nạp thành công 2619143 dòng dữ liệu gốc.

-> Đang chia tập Train/Test theo thời gian...
-> Đang thực hiện Frequency Encoding và Min-Max Scaling...

-> Đang tạo ma trận chuỗi 3D cho RF (Window = 10)...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("--- KHỞI ĐỘNG PHIÊN BẢN SPEEDTEST (100K DÒNG) ---")
file_path = "data/CIDDS-001-Sampled-Data.parquet"
df = pd.read_parquet(file_path)

# CẮT NHỎ DỮ LIỆU ĐỂ TEST NHANH (Bảo toàn trình tự thời gian)
df = df_full.iloc[:100000].copy() 
print(f"Đã cắt dữ liệu xuống còn {len(df)} dòng để test nhanh!")

# ==========================================
# 1. TIỀN XỬ LÝ LÀM SẠCH
# ==========================================
def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

# ==========================================
# 2. CHIA TẬP TRAIN/TEST (CHỐNG DATA LEAKAGE)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# ==========================================
# 3. MÃ HÓA VÀ CHUẨN HÓA AN TOÀN
# ==========================================
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

X_train, X_test = X_train.astype(float), X_test.astype(float)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)

# ==========================================
# 4. TẠO CHUỖI MULTI-FLOW ĐỘC LẬP
# ==========================================
def create_sequences(X_data, y_data, window_size=10):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - window_size):
        X_seq.append(X_data[i : i + window_size])
        y_seq.append(y_data[i + window_size])
    return np.array(X_seq), np.array(y_seq)

window_size = 10
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_encoded, window_size)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_encoded, window_size)

y_train_categorical = tf.keras.utils.to_categorical(y_train_seq, num_classes=num_classes)
y_test_categorical = tf.keras.utils.to_categorical(y_test_seq, num_classes=num_classes)

# --- GIẢI QUYẾT LỖI IN KẾT QUẢ KHI TEST TẬP NHỎ ---
# Xác định chính xác các nhãn tồn tại trong tập Test để report không bị báo lỗi thiếu nhãn
labels_in_test = np.unique(y_test_seq)
names_in_test = label_encoder.inverse_transform(labels_in_test)

# ==========================================
# 5. HUẤN LUYỆN RANDOM FOREST
# ==========================================
print("\n--- HUẤN LUYỆN RANDOM FOREST ---")
X_train_flat = X_train_seq.reshape(X_train_seq.shape[0], -1)
X_test_flat = X_test_seq.reshape(X_test_seq.shape[0], -1)

rf_multi = RandomForestClassifier(n_estimators=50, max_depth=20, max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)
rf_multi.fit(X_train_flat, y_train_seq)
rf_multi_preds = rf_multi.predict(X_test_flat)

print(f"-> F1-Score Random Forest (Multi-flow): {f1_score(y_test_seq, rf_multi_preds, average='macro'):.4f}")
print("\n[CHI TIẾT RANDOM FOREST]")
print(classification_report(y_test_seq, rf_multi_preds, labels=labels_in_test, target_names=names_in_test, digits=4, zero_division=0))

# ==========================================
# 6. HUẤN LUYỆN LSTM VỚI CLASS WEIGHT
# ==========================================
print("\n--- HUẤN LUYỆN LSTM VỚI CLASS WEIGHT ---")
unique_classes = np.unique(y_train_seq)
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=y_train_seq)
class_weight_dict = dict(zip(unique_classes, weights))
print(f"Trọng số cân bằng được gán: {class_weight_dict}")

lstm_model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(window_size, X_train_seq.shape[2]), activation='tanh'),
    Dropout(0.2),
    LSTM(100, return_sequences=False, activation='tanh'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

lstm_model.fit(
    X_train_seq, y_train_categorical, 
    epochs=5, 
    batch_size=512, 
    validation_split=0.1, 
    class_weight=class_weight_dict, 
    verbose=1
)

lstm_preds_probs = lstm_model.predict(X_test_seq)
lstm_preds = np.argmax(lstm_preds_probs, axis=1)

print(f"\n-> F1-Score LSTM (Multi-flow): {f1_score(y_test_seq, lstm_preds, average='macro'):.4f}")
print("\n[CHI TIẾT LSTM SAU KHI FIX TRỌNG SỐ]")
print(classification_report(y_test_seq, lstm_preds, labels=labels_in_test, target_names=names_in_test, digits=4, zero_division=0))

I0000 00:00:1774882070.016671  232982 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774882071.342723  232982 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774882073.502416  232982 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


--- KHỞI ĐỘNG PHIÊN BẢN SPEEDTEST (100K DÒNG) ---
Đã cắt dữ liệu xuống còn 100000 dòng để test nhanh!

--- HUẤN LUYỆN RANDOM FOREST ---
-> F1-Score Random Forest (Multi-flow): 1.0000

[CHI TIẾT RANDOM FOREST]
              precision    recall  f1-score   support

         ---     1.0000    1.0000    1.0000     29990

    accuracy                         1.0000     29990
   macro avg     1.0000    1.0000    1.0000     29990
weighted avg     1.0000    1.0000    1.0000     29990


--- HUẤN LUYỆN LSTM VỚI CLASS WEIGHT ---
Trọng số cân bằng được gán: {np.int64(0): np.float64(4.679727199786039), np.int64(1): np.float64(0.5598125159969286)}
Epoch 1/5


E0000 00:00:1774882077.961792  232982 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/luongminhthu/Documents/code/university/AI66A_CANHAN2/venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


124/124 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9796 - loss: 0.2406 - val_accuracy: 0.9939 - val_loss: 0.1902
Epoch 2/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9776 - loss: 0.2247 - val_accuracy: 0.9939 - val_loss: 0.1119
Epoch 3/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.9751 - loss: 0.2148 - val_accuracy: 0.9931 - val_loss: 0.1277
Epoch 4/5
114/124 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9752 - loss: 0.2197

KeyboardInterrupt: 

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

print("1. Đang tải và làm sạch dữ liệu...")
# df = pd.read_parquet("data/CIDDS-001-Sampled-Data.parquet") # Bỏ comment khi chạy

def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    return float(val)

# df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

print("2. Đang chia 3 tập Train / Val / Test...")
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

print("3. Đang One-Hot Encoding cho cột 'Proto'...")
X_train = pd.get_dummies(X_train, columns=['Proto'], dtype=float)
X_val   = pd.get_dummies(X_val,   columns=['Proto'], dtype=float)
X_test  = pd.get_dummies(X_test,  columns=['Proto'], dtype=float)

# Đồng bộ cột
X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("4. Đang Frequency Encoding & Chuẩn hóa...")
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True) 
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_val[col]   = X_val[col].map(freq).fillna(0)
    X_test[col]  = X_test[col].map(freq).fillna(0)

X_train, X_val, X_test = X_train.astype(float), X_val.astype(float), X_test.astype(float)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val   = label_encoder.transform(y_val)
y_test  = label_encoder.transform(y_test)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("5. Đang tính toán Trọng số thông minh (Weight Clipping)...")
# Tính trọng số phạt chuẩn
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
# TUYỆT CHIÊU: Giới hạn mức phạt tối đa là 20 để không bị "hoang tưởng"
sample_weights = np.clip(sample_weights, a_min=1.0, a_max=20.0) 

print("\n--- ĐANG TRAIN MÔ HÌNH XGBOOST CUỐI CÙNG ---")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,            
    learning_rate=0.05,          
    max_depth=7,                 
    subsample=0.8,               
    colsample_bytree=0.8,        
    tree_method='hist',          
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=20     
)

# Đưa trọng số vào để ép XGBoost học pingScan
xgb_model.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights, # <--- CHÌA KHÓA Ở ĐÂY
    eval_set=[(X_val_scaled, y_val)],  
    verbose=50                         
)

print("\n--- NGHIỆM THU TRÊN TẬP TEST TINH KHIẾT ---")
xgb_preds = xgb_model.predict(X_test_scaled)
print(f"🏆 XGBoost (Clipped Weights) F1-Score: {f1_score(y_test, xgb_preds, average='macro'):.4f}\n")
print(classification_report(y_test, xgb_preds, target_names=label_encoder.classes_, digits=4, zero_division=0))

1. Đang tải và làm sạch dữ liệu...
2. Đang chia 3 tập Train / Val / Test...
3. Đang One-Hot Encoding cho cột 'Proto'...
4. Đang Frequency Encoding & Chuẩn hóa...
5. Đang tính toán Trọng số thông minh (Weight Clipping)...

--- ĐANG TRAIN MÔ HÌNH XGBOOST CUỐI CÙNG ---
[0]	validation_0-mlogloss:0.60030
[50]	validation_0-mlogloss:0.04050
[100]	validation_0-mlogloss:0.00465
[150]	validation_0-mlogloss:0.00158
[200]	validation_0-mlogloss:0.00128
[250]	validation_0-mlogloss:0.00124
[299]	validation_0-mlogloss:0.00122

--- NGHIỆM THU TRÊN TẬP TEST TINH KHIẾT ---
🏆 XGBoost (Clipped Weights) F1-Score: 0.9135

              precision    recall  f1-score   support

         ---     0.9999    1.0000    1.0000    652723
  bruteForce     0.9973    0.9736    0.9853       379
         dos     1.0000    0.9999    0.9999    117280
    pingScan     0.5562    0.6344    0.5927       320
    portScan     0.9916    0.9878    0.9897     15041

    accuracy                         0.9996    785743
   macro avg 

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score
import xgboost as xgb
import lightgbm as lgb  # Thêm ứng cử viên mới của Microsoft!

# ==========================================
# 1. TẢI VÀ TIỀN XỬ LÝ (GIỮ NGUYÊN CHUẨN MỰC CỦA BẠN)
# ==========================================
print("1. Đang chuẩn bị dữ liệu...")
# df = pd.read_parquet("data/CIDDS-001-Sampled-Data.parquet") # Bỏ comment khi chạy

def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

# df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

# Chia tập 70/30 (Không cần Validation cho vòng Test nhanh này)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Encoding & Scaling
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])  
X_test[cat_cols]  = encoder.transform(X_test[cat_cols])       

X_train = X_train.astype(float)
X_test  = X_test.astype(float)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test  = label_encoder.transform(y_test)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled  = scaler.transform(X_test)       

# ==========================================
# 2. ĐẤU TRƯỜNG MODEL (MODEL ARENA)
# ==========================================
print("\n" + "="*50)
print("BẮT ĐẦU CHẠY THỬ NGHIỆM CÁC THUẬT TOÁN")
print("="*50)

# Khai báo các mô hình với sức mạnh tương đương nhau (100 cây)
models = {
    "1. Random Forest": RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1),
    "2. Extra Trees  ": ExtraTreesClassifier(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1),
    "3. LightGBM     ": lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "4. XGBoost      ": xgb.XGBClassifier(n_estimators=100, tree_method='hist', random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    print(f"\n[+] Đang huấn luyện {name}...")
    start_time = time.time()
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    preds = model.predict(X_test_scaled)
    
    # Đánh giá
    f1 = f1_score(y_test, preds, average='macro')
    train_time = time.time() - start_time
    
    results.append({
        "Model": name.strip("1234. "),
        "Macro F1-Score": f1,
        "Time (s)": round(train_time, 2)
    })
    
    print(f"    -> F1-Score: {f1:.4f} | Thời gian chạy: {round(train_time, 2)} giây")

# ==========================================
# 3. BẢNG TỔNG SẮP KẾT QUẢ
# ==========================================
print("\n" + "="*50)
print("🏆 BẢNG XẾP HẠNG KẾT QUẢ CHUNG CUỘC 🏆")
print("="*50)

results_df = pd.DataFrame(results).sort_values(by="Macro F1-Score", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

best_model = results_df.iloc[0]['Model']
print(f"\n=> 👑 Nhà vô địch thuộc về: {best_model}!")

1. Đang làm sạch dữ liệu...
2. Đang chia tập Train / Val / Test...
 -> Kích thước Train: 1466720 | Val: 366680 | Test: 785743
3. Đang mã hóa và chuẩn hóa (Tuyệt đối không Leakage)...

--- ĐANG TRAIN MÔ HÌNH XGBOOST ---
[0]	validation_0-mlogloss:0.44491
[20]	validation_0-mlogloss:0.11602
[40]	validation_0-mlogloss:0.04130
[60]	validation_0-mlogloss:0.01595
[80]	validation_0-mlogloss:0.00679
[100]	validation_0-mlogloss:0.00342
[120]	validation_0-mlogloss:0.00215
[140]	validation_0-mlogloss:0.00170
[160]	validation_0-mlogloss:0.00156
[180]	validation_0-mlogloss:0.00151
[200]	validation_0-mlogloss:0.00147
[220]	validation_0-mlogloss:0.00147
[234]	validation_0-mlogloss:0.00147

--- ĐÁNH GIÁ TRÊN TẬP TEST TINH KHIẾT (30% DATA) ---
XGBoost F1-Score (Macro Thực Tế): 0.8888

              precision    recall  f1-score   support

         ---     0.9998    1.0000    0.9999    652723
  bruteForce     1.0000    0.9763    0.9880       379
         dos     1.0000    0.9999    1.0000    117280
    pi

In [24]:
import pandas as pd
import glob
import os

# ==========================================
# 1. CHỈ QUÉT THƯ MỤC INTERNAL (OPENSTACK)
# ==========================================
directories = [
    "data/traffic/OpenStack_Parquet/*.parquet"
]

all_files = []
for path in directories:
    all_files.extend(glob.glob(path))

# ==========================================
# 2. CẤU HÌNH THỜI GIAN LỌC (CHUẨN BÀI BÁO)
# ==========================================
start_date = "2017-03-17 14:18:05"
end_date   = "2017-03-20 17:42:17"
time_col   = 'Date first seen' 

def load_internal_traffic(file_list, start, end):
    combined_list = []
    
    print(f"Bắt đầu quét {len(file_list)} file Parquet Internal...")
    
    for file in file_list:
        df = pd.read_parquet(file)
        
        # Đảm bảo cột thời gian đúng định dạng datetime
        df[time_col] = pd.to_datetime(df[time_col])
        
        # Lọc theo khung thời gian
        mask = (df[time_col] >= pd.to_datetime(start)) & (df[time_col] <= pd.to_datetime(end))
        filtered = df.loc[mask].copy()
        
        if not filtered.empty:
            # Gắn nhãn Internal
            filtered['Source_Type'] = "Internal"
            combined_list.append(filtered)
            print(f"  [+] {os.path.basename(file)}: Giữ lại {len(filtered):,} dòng")
            
    if combined_list:
        return pd.concat(combined_list, ignore_index=True)
    return pd.DataFrame()

# ==========================================
# 3. THỰC THI GỘP DỮ LIỆU
# ==========================================
df_internal = load_internal_traffic(all_files, start_date, end_date)

# ==========================================
# 4. KIỂM TRA VÀ LƯU KẾT QUẢ CHỐT HẠ
# ==========================================
print("\n" + "="*50)
if not df_internal.empty:
    print(f"✅ TỔNG LƯU LƯỢNG INTERNAL TÌM THẤY: {len(df_internal):,} dòng")
    print(f"Khoảng thời gian: {df_internal[time_col].min()} --> {df_internal[time_col].max()}")
    print("="*50)
    
    # LƯU RA FILE ĐỂ MANG ĐI TRAIN
    output_path = "data/CIDDS-001-Internal-Exact.parquet"
    print(f"\n5. Đang lưu tập dữ liệu chuẩn ra file: {output_path}...")
    df_internal.to_parquet(output_path, index=False)
    print("🎉 LƯU THÀNH CÔNG! Bây giờ bạn có thể mang file này đi Train mô hình rồi!")
else:
    print("❌ Không tìm thấy dữ liệu phù hợp trong thư mục.")

Bắt đầu quét 4 file Parquet Internal...
  [+] CIDDS-001-internal-week1.parquet: Giữ lại 2,535,985 dòng

✅ TỔNG LƯU LƯỢNG INTERNAL TÌM THẤY: 2,535,985 dòng
Khoảng thời gian: 2017-03-17 14:18:05 --> 2017-03-20 17:42:16.989000

5. Đang lưu tập dữ liệu chuẩn ra file: data/CIDDS-001-Internal-Exact.parquet...
🎉 LƯU THÀNH CÔNG! Bây giờ bạn có thể mang file này đi Train mô hình rồi!


In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score

# ==========================================
# PHẦN 1: TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING)
# ==========================================

# Giả sử df là dữ liệu bạn đã load
df = df_internal

print("1. Đang làm sạch cột 'Bytes'...")
def clean_bytes(val):
    if pd.isna(val):
        return 0.0
    val = str(val).strip().upper()
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

# Khai báo features (X) và target (y)
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 
    'Proto', 'Flags', 'Duration', 'Bytes', 'Packets'
]
X = df[features].copy()
y = df['attackType'].copy()

print("2. Đang chia tập Train/Test (Split 70/30)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("3. Đang mã hóa Đặc trưng (Frequency) và Nhãn (Label)...")
# A. Frequency Encoding cho X (Đầu vào)
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

# Ép kiểu float chuẩn bị cho Scaling
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# B. Label Encoding cho y (Đầu ra)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n[+] Hoàn tất Tiền xử lý! Kích thước dữ liệu:")
print(f" - Train set: {X_train_scaled.shape}")
print(f" - Test set: {X_test_scaled.shape}")
print(f" - Từ điển Nhãn (y): {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


# ==========================================
# PHẦN 2: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH
# ==========================================
print("\n--- 5. ĐANG TRAIN LẠI MÔ HÌNH ---")

# Random Forest: Đã tăng n_estimators và thêm class_weight='balanced'
rf_model_tuned = RandomForestClassifier(
    n_estimators=10,            # Tăng từ 10 lên 100 cây để chống nhiễu
    criterion='entropy',            
    min_samples_split=2,         
    min_samples_leaf=1,          
    max_features='sqrt',         
    class_weight=None,     # Cứu cánh cho các lớp thiểu số
    random_state=42,             
    n_jobs=-1                    
)
rf_model_tuned.fit(X_train_scaled, y_train)
rf_preds_tuned = rf_model_tuned.predict(X_test_scaled)

# KNN giữ nguyên tham số
knn_model_tuned = KNeighborsClassifier(
    n_neighbors=3, weights='distance', leaf_size=30, metric='minkowski', n_jobs=-1
)
knn_model_tuned.fit(X_train_scaled, y_train)
knn_preds_tuned = knn_model_tuned.predict(X_test_scaled)

print("\n--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---")
print(f"Random Forest F1-Score: {f1_score(y_test, rf_preds_tuned, average='macro'):.4f}")
print(f"KNN F1-Score:           {f1_score(y_test, knn_preds_tuned, average='macro'):.4f}")

print("\n--- CHI TIẾT BÁO CÁO CỦA RANDOM FOREST ---")
print(classification_report(y_test, rf_preds_tuned, target_names=label_encoder.classes_, digits=4, zero_division=0))

1. Đang làm sạch cột 'Bytes'...
2. Đang chia tập Train/Test (Split 70/30)...
3. Đang mã hóa Đặc trưng (Frequency) và Nhãn (Label)...
4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...

[+] Hoàn tất Tiền xử lý! Kích thước dữ liệu:
 - Train set: (1775189, 9)
 - Test set: (760796, 9)
 - Từ điển Nhãn (y): {'---': np.int64(0), 'bruteForce': np.int64(1), 'dos': np.int64(2), 'pingScan': np.int64(3), 'portScan': np.int64(4)}

--- 5. ĐANG TRAIN LẠI MÔ HÌNH ---

--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---
Random Forest F1-Score: 0.9295
KNN F1-Score:           0.9269

--- CHI TIẾT BÁO CÁO CỦA RANDOM FOREST ---
              precision    recall  f1-score   support

         ---     0.9999    1.0000    1.0000    627776
  bruteForce     1.0000    0.9868    0.9934       379
         dos     1.0000    0.9999    1.0000    117280
    pingScan     0.8302    0.5500    0.6617       320
    portScan     0.9898    0.9958    0.9928     15041

    accuracy                         0.9997    760796
   macro av